In [ ]:
# 셀 1
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 셀 2 — 프로젝트 폴더를 코랩 로컬로 복사 후 이동
import shutil, os
shutil.copytree('/content/drive/MyDrive/LPRNet-master2', '/content/LPRNet-master')
os.chdir('/content/LPRNet-master')


In [ ]:
# 셀 3 — 데이터셋 압축 해제 (경로 변경 없음)
!unzip -q ALPRData.zip


In [ ]:
# 셀 4 — 의존성 설치
!pip install lightning opencv-python-headless pyyaml tqdm rich albumentations -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 62.6 MB/s eta 0:00:00


In [ ]:
!grep -n "max_epochs\|patience" /content/LPRNet-master/train.py


58:        max_epochs=300,
74:                patience=30,


In [ ]:
# 셀 5 — 학습
!python train.py


[1/4] 설정 로드 완료
  train_dir : ALPRData/dataset/train/
  valid_dir : ALPRData/dataset/val/
  batch_size: 256
  lr        : 0.001
  ckpt 저장 : saving_ckpt_05-11_02:03
  device    : Tesla T4
[2/4] 사전학습 가중치 로드: weights/lprnet_kor.pt
dm loaded
[3/4] 데이터 로드 완료
[4/4] 학습 시작
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
train:  6757
val:  1447
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
┏━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ STNet  │ _STNet  │  118 K │ train │     0 │
│ 1 │ LPRNet │ _LPRNet │  988 K │ train │     0 │
└───┴────────┴─────────┴────────┴───────┴───────┘
Trainable param

In [ ]:
import shutil
shutil.copytree(
    '/content/LPRNet-master/saving_ckpt_05-11_02:03',
    '/content/drive/MyDrive/saving_ckpt_05-11_02:03',
    dirs_exist_ok=True
)
print("체크포인트 Drive 백업 완료")


체크포인트 Drive 백업 완료


In [ ]:
# 셀 6 — 체크포인트 Drive에 백업
import glob
for d in glob.glob('saving_ckpt*'):
    shutil.copytree(d, f'/content/drive/MyDrive/{d}')


In [ ]:
import shutil, glob

src = '/content/LPRNet-master/saving_ckpt_05-01_04:22'
dst = '/content/drive/MyDrive/LPRNet-master/saving_ckpt_05-01_04:22'
shutil.copytree(src, dst, dirs_exist_ok=True)
print("백업 완료")

# 저장된 체크포인트 목록 확인
for f in sorted(glob.glob(src + '/*.ckpt')):
    print(f)


백업 완료
/content/LPRNet-master/saving_ckpt_05-01_04:22/epoch=134-val-acc=0.974.ckpt
/content/LPRNet-master/saving_ckpt_05-01_04:22/epoch=137-val-acc=0.972.ckpt
/content/LPRNet-master/saving_ckpt_05-01_04:22/epoch=146-val-acc=0.974.ckpt
/content/LPRNet-master/saving_ckpt_05-01_04:22/epoch=147-val-acc=0.973.ckpt
/content/LPRNet-master/saving_ckpt_05-01_04:22/epoch=149-val-acc=0.974.ckpt
/content/LPRNet-master/saving_ckpt_05-01_04:22/last.ckpt


In [ ]:
!pip install onnxscript onnx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.0 MB/s eta 0:00:00


In [ ]:
import torch
from lprnet import LPRNet

ckpt_path = "saving_ckpt_05-11_02:03/epoch=277-val-acc=0.988.ckpt"
model = LPRNet.load_from_checkpoint(ckpt_path, map_location="cpu")
model.eval()

dummy_input = torch.randn(1, 3, 50, 100)
torch.onnx.export(
    model,
    dummy_input,
    "lprnet_kor_finetuned4.onnx",
    opset_version=18,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)
print("ONNX 저장 완료")

# 추론 검증
!pip install onnxruntime -q
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession("lprnet_kor_finetuned4.onnx", providers=["CPUExecutionProvider"])
dummy_np = dummy_input.numpy()
result = sess.run(["output"], {"input": dummy_np})
print(f"추론 검증 완료 — output shape: {result[0].shape}")

/tmp/ipykernel_6441/1224931971.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0511 03:20:30.952000 6441 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0511 03:20:30.954000 6441 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0511 03:20:30.956000 6441 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: '

[torch.onnx] Obtain model graph for `LPRNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `LPRNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX 저장 완료
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.3 MB/s eta 0:00:00
추론 검증 완료 — output shape: (1, 110, 19)


In [ ]:
import shutil
import os

# 저장할 드라이브 경로 (원하는 폴더명으로 변경 가능)
save_dir = '/content/drive/MyDrive/lprnet_onnx'
os.makedirs(save_dir, exist_ok=True)

# 복사할 파일 목록
files = [
    'lprnet_kor_finetuned4.onnx',
    'lprnet_kor_finetuned4.onnx.data'
]

for f in files:
    shutil.copy(f, os.path.join(save_dir, f))
    print(f'✅ {f} 저장 완료')

print(f'\n전체 저장 경로: {save_dir}')

✅ lprnet_kor_finetuned4.onnx 저장 완료
✅ lprnet_kor_finetuned4.onnx.data 저장 완료

전체 저장 경로: /content/drive/MyDrive/lprnet_onnx


In [ ]:
import os
print(os.listdir("/content/LPRNet-master/ALPRData"))


['dataset']


In [ ]:
!pip install onnxruntime -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 99.7 MB/s eta 0:00:00


In [ ]:
import onnxruntime as ort
import numpy as np
import cv2, os, re, unicodedata, yaml

with open("/content/LPRNet-master/config/kor_config.yaml") as f:
    cfg = yaml.full_load(f)
chars = cfg['chars']

sess = ort.InferenceSession("/content/LPRNet-master/lprnet_onnx/lprnet_kor_finetuned4.onnx")

def preprocess(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (100, 50), interpolation=cv2.INTER_CUBIC)
    img = img.astype("float32")
    img -= 127.5
    img *= 0.0078125          # app.py와 동일한 정규화
    return np.transpose(img, (2, 0, 1))[np.newaxis]  # (1, 3, 50, 100)

def decode_ctc(out, chars):
    pred = out[0][0]          # (110, 19)
    blank = len(chars) - 1   # index 109 = blank 토큰 ('중')
    result, prev = [], -1
    for j in range(pred.shape[1]):
        c = int(np.argmax(pred[:, j]))
        if c != prev and c != blank:   # blank 및 반복 제거
            result.append(chars[c])
        prev = c
    return ''.join(result)


test_dir = "/content/LPRNet-master/ALPRData/dataset/test"  # test 폴더 있으면 변경
files = [f for f in sorted(os.listdir(test_dir)) if f.endswith('.jpg')]
total, correct = 0, 0

for f in files:
    label = unicodedata.normalize('NFC', re.sub(r'_\d+\.jpg$', '', f))
    out = sess.run(None, {"input": preprocess(os.path.join(test_dir, f))})
    pred = unicodedata.normalize('NFC', decode_ctc(out, chars))
    total += 1
    if label == pred:
        correct += 1
    print(f"[{total:4d}] 정답: {label} | 추론: {pred} | {'O' if label==pred else 'X'}")

print(f"\n총 {total}장 | 정답 {correct}장 | 정확도: {correct/total*100:.2f}%")


[   1] 정답: 100누1753 | 추론: 100누1753 | O
[   2] 정답: 102누5329 | 추론: 102누5329 | O
[   3] 정답: 103버3634 | 추론: 103버3634 | O
[   4] 정답: 108거6275 | 추론: 108거6275 | O
[   5] 정답: 10마8110 | 추론: 10마8110 | O
[   6] 정답: 10머9440 | 추론: 10머9440 | O
[   7] 정답: 10부2112 | 추론: 10부2112 | O
[   8] 정답: 10오8497 | 추론: 10오8497 | O
[   9] 정답: 10오8497 | 추론: 10오8497 | O
[  10] 정답: 10저9580 | 추론: 10저9580 | O
[  11] 정답: 114너2892 | 추론: 114너2892 | O
[  12] 정답: 114보8677 | 추론: 114보8677 | O
[  13] 정답: 117더5325 | 추론: 117더5325 | O
[  14] 정답: 119가5880 | 추론: 119가5880 | O
[  15] 정답: 11너5133 | 추론: 11너5133 | O
[  16] 정답: 11로7877 | 추론: 11로7877 | O
[  17] 정답: 11보9604 | 추론: 11보9604 | O
[  18] 정답: 11저8592 | 추론: 11저8592 | O
[  19] 정답: 124아7554 | 추론: 124아7554 | O
[  20] 정답: 127허0456 | 추론: 127허0456 | O
[  21] 정답: 127허0456 | 추론: 127허0456 | O
[  22] 정답: 12러4240 | 추론: 12러4240 | O
[  23] 정답: 12무1136 | 추론: 12무1136 | O
[  24] 정답: 12버3517 | 추론: 12버3517 | O
[  25] 정답: 12호6894 | 추론: 12호6894 | O
[  26] 정답: 12호6894 | 추론: 12호6894 | O
[  27] 정답: 137저1